In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Bina Circuit

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
```
</details>
Halaman ini mengkaji dengan lebih mendalam kelas [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) dalam SDK Qiskit, termasuk beberapa kaedah yang lebih lanjut yang boleh anda gunakan untuk mencipta Circuit kuantum.
## Apakah Circuit kuantum?
Circuit kuantum mudah ialah koleksi Qubit dan senarai arahan yang bertindak ke atas Qubit tersebut. Untuk menunjukkan, sel berikut mencipta Circuit baharu dengan dua Qubit baharu, kemudian memaparkan atribut [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) Circuit, yang merupakan senarai [`Qubit`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) dalam susunan dari bit paling tidak bererti $q_0$ hingga bit paling bererti $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

Berbilang objek `QuantumRegister` dan `ClassicalRegister` boleh digabungkan untuk mencipta sebuah Circuit. Setiap [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) dan [`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) juga boleh dinamakan.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

Anda boleh mencari indeks dan daftar Qubit dengan menggunakan kaedah [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) Circuit dan atributnya.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

Menambah arahan kepada Circuit menambahkan arahan tersebut ke atribut [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) Circuit. Output sel berikut menunjukkan `data` ialah senarai objek [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction), setiap satunya mempunyai atribut `operation`, dan atribut `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

Cara termudah untuk melihat maklumat ini ialah melalui kaedah [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw), yang mengembalikan visualisasi Circuit. Lihat [Visualisasikan Circuit](./visualize-circuits) untuk cara berbeza memaparkan Circuit kuantum.

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

Objek arahan Circuit boleh mengandungi Circuit "definisi" yang menggambarkan arahan dari segi arahan yang lebih asas. Contohnya, [X-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) ditakrifkan sebagai kes tertentu [U3-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate), sebuah Gate Qubit tunggal yang lebih umum.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

Arahan dan Circuit adalah serupa kerana kedua-duanya menerangkan operasi pada bit dan Qubit, tetapi mempunyai tujuan yang berbeza:

- Arahan dianggap tetap, dan kaedahnya biasanya mengembalikan arahan baharu (tanpa mengubah objek asal).
- Circuit direka untuk dibina merentasi banyak baris kod, dan kaedah [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) sering mengubah objek yang sedia ada.
### Apakah kedalaman Circuit?
[Kedalaman (depth())](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) Circuit kuantum ialah ukuran bilangan "lapisan" Gate kuantum, yang dilaksanakan secara selari, yang diperlukan untuk melengkapkan pengiraan yang ditakrifkan oleh Circuit tersebut. Oleh kerana Gate kuantum mengambil masa untuk dilaksanakan, kedalaman Circuit lebih kurang sepadan dengan jumlah masa yang diambil oleh komputer kuantum untuk melaksanakan Circuit tersebut. Oleh itu, kedalaman Circuit merupakan salah satu kuantiti penting yang digunakan untuk mengukur sama ada Circuit kuantum boleh dijalankan pada peranti.

Selebihnya halaman ini menggambarkan cara memanipulasi Circuit kuantum.
## Bina Circuit
Kaedah seperti [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) dan [`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) menambah arahan tertentu kepada Circuit. Untuk menambah arahan kepada Circuit secara lebih umum, gunakan kaedah [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). Ini mengambil arahan dan senarai Qubit untuk menerapkan arahan tersebut. Lihat [dokumentasi API Pustaka Circuit](https://docs.quantum.ibm.com/api/qiskit/circuit_library) untuk senarai arahan yang disokong.

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

Untuk menggabungkan dua Circuit, gunakan kaedah [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). Ini menerima [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) lain dan senarai pemetaan Qubit yang pilihan.

> **Note:** Kaedah [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) mengembalikan Circuit baharu dan **tidak** mengubah mana-mana Circuit yang ia bertindak ke atasnya. Untuk mengubah Circuit yang anda panggil kaedah [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose), gunakan argumen `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

Untuk melihat apa yang berlaku, anda boleh menggunakan kaedah [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) untuk mengembangkan setiap arahan ke dalam definisinya.

> **Note:** Kaedah [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) mengembalikan Circuit baharu dan **tidak** mengubah Circuit yang ia bertindak ke atasnya.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## Ukur Qubit
Pengukuran digunakan untuk menyampel keadaan Qubit individu dan memindahkan keputusan ke daftar klasik. Perlu diingat bahawa jika anda menghantar Circuit kepada primitif [Sampler](./primitives#sampler), pengukuran diperlukan. Walau bagaimanapun, Circuit yang dihantar kepada primitif [Estimator](./primitives#estimator) tidak boleh mengandungi pengukuran.

Qubit boleh diukur menggunakan tiga kaedah: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) dan [`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). Untuk mempelajari cara menvisualisasikan keputusan yang diukur, lihat halaman [Visualisasikan keputusan](./visualize-results).

1. `QuantumCircuit.measure` : mengukur setiap Qubit dalam argumen pertama ke atas bit klasik yang diberikan sebagai argumen kedua. Kaedah ini membolehkan kawalan penuh ke atas tempat keputusan pengukuran disimpan.

2. `QuantumCircuit.measure_all` : tidak mengambil argumen dan boleh digunakan untuk Circuit kuantum tanpa bit klasik yang telah ditakrifkan. Ia mencipta wayar klasik dan menyimpan keputusan pengukuran mengikut susunan. Contohnya, pengukuran Qubit $q_i$ disimpan dalam cbit $meas_i$). Ia juga menambah halangan sebelum pengukuran.

3. `QuantumCircuit.measure_active` : serupa dengan `measure_all`, tetapi hanya mengukur Qubit yang mempunyai operasi.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output of the previous code cell](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## Circuit berparameter

Banyak algoritma kuantum jangka pendek melibatkan pelaksanaan banyak variasi Circuit kuantum. Oleh kerana membina dan mengoptimumkan Circuit yang besar boleh memakan masa secara komputasi, Qiskit menyokong Circuit **berparameter**. Circuit ini mempunyai parameter yang tidak ditakrifkan, dan nilainya tidak perlu ditakrifkan sehingga sejurus sebelum melaksanakan Circuit. Ini membolehkan anda memindahkan pembinaan dan pengoptimuman Circuit keluar daripada gelung program utama. Sel berikut mencipta dan memaparkan Circuit berparameter.